# DCL Eval Pipeline — Demo

**Evaluation pipeline for monitoring and auditing LLM agent behavior in multi-agent systems.**

This notebook demonstrates:
- Prompt quality assessment (clarity, completeness)
- Response consistency checks
- Anomaly detection in agent decision chains
- Structured audit logging

---
> Running via **HuggingFace Inference API** (free tier) — no paid API key required.

> Intended audience: AI leads / architects in fintech & gov who need a concrete evaluation layer for agent behavior audit.

## 0. Setup

Requires a free HuggingFace account and token:
1. Register at https://huggingface.co
2. Go to **Settings → Access Tokens → New token** (Read is enough)
3. Paste your token below

In [ ]:
import json
import sys
import os
from datetime import datetime
from dataclasses import asdict
import requests

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

# --- Paste your HuggingFace token here ---
HF_TOKEN = "hf_your_token_here"  # ← replace with your token
# -----------------------------------------

MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
HF_URL = f"https://api-inference.huggingface.co/models/{MODEL}"

def call_llm(prompt: str) -> str:
    """Call HuggingFace Inference API (free tier)."""
    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    payload = {
        "inputs": f"<s>[INST] {prompt} [/INST]",
        "parameters": {
            "max_new_tokens": 512,
            "temperature": 0.3,
            "return_full_text": False
        }
    }
    response = requests.post(HF_URL, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()[0]["generated_text"].strip()

# Verify connection
test = call_llm("Say 'OK' and nothing else.")
print(f"✓ HuggingFace connected | Model: {MODEL}")
print(f"  Response: {test[:80]}")

## 1. Metrics Engine

Three core metrics from `eval/metrics.py`:
- **Consistency** — keyword matching against expected output
- **Length** — response within acceptable bounds
- **Anomaly** — forbidden pattern detection

In [ ]:
from eval.metrics import score_consistency, score_length, score_anomaly, EvalResult

print("✓ Metrics loaded: score_consistency, score_length, score_anomaly")

## 2. Prompt Templates

Three evaluation scenarios from `prompts/templates.py`:
- `basic_reasoning` — step-by-step task solving
- `decision_audit` — decision chain traceability  
- `anomaly_detection` — log entry analysis

In [ ]:
from prompts.templates import get_prompt, AGENT_EVAL_PROMPTS

print("Available templates:")
for name in AGENT_EVAL_PROMPTS:
    print(f"  → {name}")

## 3. Evaluation Pipeline (Ollama version)

Adapted `pipeline.py` to run on local Llama 3.2 instead of OpenAI.

In [ ]:
def run_eval_local(template_name: str, **template_kwargs) -> dict:
    """Run evaluation pipeline using local Ollama model."""
    
    template = get_prompt(template_name, **template_kwargs)
    prompt = template["prompt"]
    expected_keywords = template["expected_keywords"]
    forbidden_patterns = template["forbidden_patterns"]
    
    # Get response from local LLM
    text = call_llm(prompt)
    
    # Run metrics
    metrics = [
        asdict(score_consistency(text, expected_keywords)),
        asdict(score_length(text)),
        asdict(score_anomaly(text, forbidden_patterns))
    ]
    
    # Compute aggregate score
    aggregate = round(sum(m["score"] for m in metrics) / len(metrics), 2)
    passed_all = all(m["passed"] for m in metrics)
    
    result = {
        "timestamp": datetime.utcnow().isoformat(),
        "template": template_name,
        "prompt": prompt,
        "response": text,
        "aggregate_score": aggregate,
        "passed": passed_all,
        "metrics": metrics
    }
    
    # Append to log
    log_path = "../data/sample_logs.json"
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")
    
    return result

print("✓ Pipeline ready")

## 4. Run: Basic Reasoning Audit

Test whether the agent can produce traceable step-by-step reasoning.

In [ ]:
result_1 = run_eval_local(
    "basic_reasoning",
    task="Classify a loan application as high or low risk based on income and credit history"
)

print(f"Template : {result_1['template']}")
print(f"Score    : {result_1['aggregate_score']} | Passed: {result_1['passed']}")
print(f"\nResponse preview:\n{result_1['response'][:400]}...")
print("\n--- Metrics ---")
for m in result_1["metrics"]:
    status = "✓" if m["passed"] else "✗"
    print(f"{status} {m['prompt_id']}: {m['score']} — {m['reason']}")

## 5. Run: Decision Chain Audit

Can the agent provide an auditable decision chain? Critical for compliance scenarios.

In [ ]:
result_2 = run_eval_local(
    "decision_audit",
    scenario="Whether to flag a transaction of $47,000 from a new account with no prior history"
)

print(f"Template : {result_2['template']}")
print(f"Score    : {result_2['aggregate_score']} | Passed: {result_2['passed']}")
print(f"\nResponse preview:\n{result_2['response'][:400]}...")
print("\n--- Metrics ---")
for m in result_2["metrics"]:
    status = "✓" if m["passed"] else "✗"
    print(f"{status} {m['prompt_id']}: {m['score']} — {m['reason']}")

## 6. Run: Anomaly Detection — Injected Failure

We inject a log entry designed to trigger anomaly flags.
The pipeline should detect the forbidden pattern and score accordingly.

In [ ]:
# This log entry contains signals that should trigger anomaly detection
suspicious_log = """
agent_id: scorer_v2 | step: 7 | step: 7 | step: 7
action: evaluate_kyc
reasoning: skipping verification — not important for this case
result: approved
"""

result_3 = run_eval_local(
    "anomaly_detection",
    log_entry=suspicious_log
)

print(f"Template : {result_3['template']}")
print(f"Score    : {result_3['aggregate_score']} | Passed: {result_3['passed']}")
print(f"\nAgent analysis:\n{result_3['response'][:500]}...")
print("\n--- Metrics ---")
for m in result_3["metrics"]:
    status = "✓" if m["passed"] else "✗"
    print(f"{status} {m['prompt_id']}: {m['score']} — {m['reason']}")

## 7. Multi-Run Summary

Aggregate view across all evaluation runs — as it would appear in a compliance audit report.

This is what your compliance officer or external auditor would see as a high-level summary.

In [ ]:
results = [result_1, result_2, result_3]

print("=" * 60)
print("DCL EVAL PIPELINE — AUDIT SUMMARY")
print(f"Generated: {datetime.utcnow().isoformat()}")
print(f"Model : {MODEL}")
print("=" * 60)

for r in results:
    status = "PASS" if r["passed"] else "FAIL"
    flag = "🔴" if not r["passed"] else "🟢"
    print(f"{flag} [{status}] {r['template']:<20} score={r['aggregate_score']}")

print("-" * 60)
overall = round(sum(r["aggregate_score"] for r in results) / len(results), 2)
failed = [r["template"] for r in results if not r["passed"]]
print(f"Overall score : {overall}")
print(f"Failed chains : {failed if failed else 'none'}")
print(f"Logs saved to : data/sample_logs.json")
print("=" * 60)

## 8. Inspect Audit Log

Every run is appended to `data/sample_logs.json` — structured, traceable, ready for downstream compliance tooling.

In [ ]:
log_path = "../data/sample_logs.json"

entries = []
with open(log_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            entries.append(json.loads(line))

print(f"Total logged entries: {len(entries)}")
print("\nLast entry (truncated):")
last = entries[-1].copy()
last["response"] = last["response"][:150] + "..."
print(json.dumps(last, indent=2, ensure_ascii=False))

---

## Summary

This demo shows the **public evaluation layer** of the DCL pipeline:

| Component | Status |
|---|---|
| Prompt template engine | ✓ |
| LLM-as-a-judge (local, no API key) | ✓ |
| Consistency / Length / Anomaly metrics | ✓ |
| Structured audit logging | ✓ |
| Aggregate scoring + pass/fail | ✓ |

**Core DCL architecture** (deterministic commitment layer, cryptographic audit trail, multi-agent consistency engine) is under IP protection.

→ [github.com/DariRinch](https://github.com/DariRinch)